# Cotizaciones — extracción y merge

4 queries a Amplitude + merges en pandas para llegar a una fila por cotización con `dni` y `resolution`.

| # | Evento | Trae | Llave |
|---|---|---|---|
| 1 | `Product Updated` | lead_id, sales_channel, brand, model, color, price_net | — |
| 2 | `Personal Information Completed` | lead_id, dni | join por lead_id |
| 3 | `Deal Created` | lead_id, deal_id (puente) | join por lead_id |
| 4 | `Deal Assigned` | deal_id, resolution | join por deal_id |

`Deal Assigned` no tiene `lead_id`, por eso usamos `Deal Created` como puente.

In [9]:
import sys
sys.path.append("../../")
import os
import json
import base64
import requests
import pandas as pd
from datetime import date, timedelta
from src.utils.amplitude import *


from dotenv import load_dotenv

load_dotenv()

True

In [10]:
# --- Config global ---

fecha_corte = date.today()
dias_ventana = 12
dias_bucket = 3

# Filtros base. sales_channel se filtra a nivel evento (igual que price_diff).
# Si Amplitude rechaza alguno, probar con subprop_type: "user".
FILTROS_BASE = [
    {"subprop_type": "event", "subprop_key": "country_code",  "subprop_op": "is", "subprop_value": ["MX"]},
    {"subprop_type": "event", "subprop_key": "sales_channel", "subprop_op": "is", "subprop_value": ["dealer"]},
]

print("fecha_corte:", fecha_corte, "| ventana:", dias_ventana, "dias")

fecha_corte: 2026-05-27 | ventana: 12 dias


## 1. `Product Updated` — cotizaciones

Group by (todos event props): `lead_id`, `sales_channel`, `brand`, `model`, `color`, `price_net`

In [11]:
event_product = {
    "event_type": "Product Updated",
    "filters": FILTROS_BASE,
    "group_by": [
        {"type": "event", "value": "lead_id"},
        {"type": "event", "value": "sales_channel"},
        {"type": "event", "value": "brand"},
        {"type": "event", "value": "model"},
        {"type": "event", "value": "color"},
        {"type": "event", "value": "price_net"},
    ],
}
gb_product = [g["value"] for g in event_product["group_by"]]

print("Extrayendo Product Updated...")
df_product = extract_event(event_product, gb_product, fecha_corte, dias_ventana, dias_bucket)
print(f"\nTotal filas: {len(df_product)}")
print(f"lead_id unicos: {df_product['lead_id'].nunique() if 'lead_id' in df_product.columns else 'n/a'}")
df_product.head()

Extrayendo Product Updated...
event_type: Product Updated
group_by enviado: [{'type': 'event', 'value': 'lead_id'}, {'type': 'event', 'value': 'sales_channel'}, {'type': 'event', 'value': 'brand'}, {'type': 'event', 'value': 'model'}, {'type': 'event', 'value': 'color'}, {'type': 'event', 'value': 'price_net'}]
  [parser] label[0] type=list repr=[0, '6a089fc6c06ef2092619084e; dealer; TVS; Stryker 3V; Rojo; 32990']
  2026-05-16 -> 2026-05-18: 6658 series, 6668 filas
  [parser] label[0] type=list repr=[0, '6a0900354634adcff48ea2bb; dealer; Zmoto; ZM 150; (none); 23499']
  2026-05-19 -> 2026-05-21: 9541 series, 9563 filas  <-- WARNING: cerca del limite de 10k
  [parser] label[0] type=list repr=[0, '6a11f4740cfc988b164334a1; dealer; Bajaj; Dominar 250; (none); 71499']
  2026-05-22 -> 2026-05-24: 6517 series, 6527 filas
  [parser] label[0] type=list repr=[0, '6a15cd3427eb15cbc4c3aac0; dealer; Vento; Crossmax 220; Rojo; 29699']
  2026-05-25 -> 2026-05-27: 7466 series, 7475 filas

Total filas

,date,lead_id,sales_channel,brand,model,color,price_net,totals
0,2026-05-16,6a089fc6c06ef2092619084e,dealer,TVS,Stryker 3V,Rojo,32990,4
1,2026-05-16,6a08c402decb411f4378bb24,dealer,Bajaj,Pulsar N 160,(none),37099,2
2,2026-05-18,6a08c402decb411f4378bb24,dealer,Bajaj,Pulsar N 160,(none),37099,2
3,2026-05-18,6a0b427a48fd1283afd97ea8,dealer,Loong,GR300,(none),55000,4
4,2026-05-18,6a0b4e3a86cb1ab4b5ccafb8,dealer,RYDER,RD-2.5 250,Rojo,41999,4


## 2. `Personal info completed` — DNI por lead

Aquí es donde se crea el `dni` por primera vez en el funnel.

Group by: `lead_id`, `dni`

In [12]:
event_personal = {
    "event_type": "Personal Information Completed",
    "filters": FILTROS_BASE,
    "group_by": [
        {"type": "event", "value": "lead_id"},
        {"type": "event", "value": "dni"},
    ],
}
gb_personal = [g["value"] for g in event_personal["group_by"]]

print("Extrayendo Personal Information Completed...")
df_personal = extract_event(event_personal, gb_personal, fecha_corte, dias_ventana, dias_bucket)
print(f"\nTotal filas: {len(df_personal)}")
print(f"lead_id unicos: {df_personal['lead_id'].nunique() if 'lead_id' in df_personal.columns else 'n/a'}")
df_personal.head()

Extrayendo Personal Information Completed...
event_type: Personal Information Completed
group_by enviado: [{'type': 'event', 'value': 'lead_id'}, {'type': 'event', 'value': 'dni'}]
  [parser] label[0] type=list repr=[0, '6a0b43d7ed065ccc38314870; AUMM070417HCMNRRA0']
  2026-05-16 -> 2026-05-18: 5189 series, 5190 filas
  [parser] label[0] type=list repr=[0, '6a0ea61cdb72530c7a379716; VILC040111HJCLPRA6']
  2026-05-19 -> 2026-05-21: 7526 series, 7528 filas
  [parser] label[0] type=list repr=[0, '6a107b5710654ba0f1f1c79e; ROLL850306HMCDDS00']
  2026-05-22 -> 2026-05-24: 5100 series, 5100 filas
  [parser] label[0] type=list repr=[0, '6a12000a855f73669830767d; COML950909HMCRRN08']
  2026-05-25 -> 2026-05-27: 5819 series, 5819 filas

Total filas: 23637
lead_id unicos: 23580


,date,lead_id,dni,totals
0,2026-05-18,6a0b43d7ed065ccc38314870,AUMM070417HCMNRRA0,3
1,2026-05-18,6a06324b004ede0247b91789,MASJ970915HMCLRN06,2
2,2026-05-16,6a089054c06ef2092618e7d3,MAPG010722HMNNRBA2,2
3,2026-05-16,6a0897244634adcff48dbf64,MOAA040811HMSRLNA3,2
4,2026-05-16,6a089c6cdecb411f43784e94,MACM070112HDFLSXA2,2


## 3. `Deal Created` + `Deal Assigned`

`Deal Assigned` no tiene `lead_id`, asi que necesitamos `Deal Created` como puente:

- **Deal Created** → `lead_id`, `deal_id`
- **Deal Assigned** → `deal_id`, `resolution`

In [13]:
# Deal Created: puente lead_id <-> deal_id
event_deal_created = {
    "event_type": "Deal Created",
    "filters": FILTROS_BASE,
    "group_by": [
        {"type": "event", "value": "lead_id"},
        {"type": "event", "value": "deal_id"},
    ],
}
gb_dc = [g["value"] for g in event_deal_created["group_by"]]

print("Extrayendo Deal Created...")
df_deal_created = extract_event(event_deal_created, gb_dc, fecha_corte, dias_ventana, dias_bucket)
print(f"\nDeal Created: {len(df_deal_created)} filas, lead_id unicos: {df_deal_created['lead_id'].nunique() if 'lead_id' in df_deal_created.columns else 'n/a'}")

# Deal Assigned: deal_id -> resolution
event_deal_assigned = {
    "event_type": "Deal Assigned",
    "filters": FILTROS_BASE,
    "group_by": [
        {"type": "event", "value": "deal_id"},
        {"type": "event", "value": "resolution"},
    ],
}
gb_da = [g["value"] for g in event_deal_assigned["group_by"]]

print("\nExtrayendo Deal Assigned...")
df_deal_assigned = extract_event(event_deal_assigned, gb_da, fecha_corte, dias_ventana, dias_bucket)
print(f"\nDeal Assigned: {len(df_deal_assigned)} filas, deal_id unicos: {df_deal_assigned['deal_id'].nunique() if 'deal_id' in df_deal_assigned.columns else 'n/a'}")
df_deal_assigned.head()

Extrayendo Deal Created...
event_type: Deal Created
group_by enviado: [{'type': 'event', 'value': 'lead_id'}, {'type': 'event', 'value': 'deal_id'}]
  [parser] label[0] type=list repr=[0, '6a078ff3366ff3e34000359e; 6a08b8965e82c30ee668f9c9']
  2026-05-16 -> 2026-05-18: 4706 series, 4706 filas
  [parser] label[0] type=list repr=[0, '6a076e54b1b886c4746bad08; 6a0fb4fe20a6c1a1c1a2b78d']
  2026-05-19 -> 2026-05-21: 6869 series, 6869 filas
  [parser] label[0] type=list repr=[0, '6a0cbe6286397c35264f60ce; 6a11ca540fd8dd06f42f77c5']
  2026-05-22 -> 2026-05-24: 4657 series, 4657 filas
  [parser] label[0] type=list repr=[0, '6a0de919b4efa54481e76aca; 6a14b73975be08971899682f']
  2026-05-25 -> 2026-05-27: 5282 series, 5282 filas

Deal Created: 21514 filas, lead_id unicos: 21513

Extrayendo Deal Assigned...
event_type: Deal Assigned
group_by enviado: [{'type': 'event', 'value': 'deal_id'}, {'type': 'event', 'value': 'resolution'}]
  [parser] label[0] type=list repr=[0, '6a07ec515b60da0a86fdef62; 

,date,deal_id,resolution,totals
0,2026-05-16,6a07ec515b60da0a86fdef62,rejected,1
1,2026-05-16,6a07ef9571a566306e5f50d5,rejected,1
2,2026-05-16,6a07eff3bd8d6f5b3f0a61b0,rejected,1
3,2026-05-16,6a07f0b6c23460deed4a5c6b,approved,1
4,2026-05-16,6a07f210a0aeabca893db712,rejected,1


## 4. Merge

- `df_product` ⟕ `df_personal` por `lead_id` → agrega `dni`
- resultado ⟕ `df_deal` por `lead_id` → agrega `deal_id`, `resolution`

Grano final: 1 fila por cotización.

In [14]:
# 1) dni por lead_id
personal_cols = [c for c in ["lead_id", "dni"] if c in df_personal.columns]
personal_dedup = (
    df_personal[personal_cols]
    .dropna(subset=["lead_id"])
    .drop_duplicates(subset=["lead_id"], keep="first")
)

# 2) puente lead_id -> deal_id (Deal Created)
dc_cols = [c for c in ["lead_id", "deal_id"] if c in df_deal_created.columns]
deal_created_dedup = (
    df_deal_created[dc_cols]
    .dropna(subset=["lead_id"])
    .drop_duplicates(subset=["lead_id"], keep="first")
)

# 3) resolution por deal_id (Deal Assigned)
da_cols = [c for c in ["deal_id", "resolution"] if c in df_deal_assigned.columns]
deal_assigned_dedup = (
    df_deal_assigned[da_cols]
    .dropna(subset=["deal_id"])
    .drop_duplicates(subset=["deal_id"], keep="first")
)

df_final = (
    df_product
    .merge(personal_dedup, on="lead_id", how="left")
    .merge(deal_created_dedup, on="lead_id", how="left")
    .merge(deal_assigned_dedup, on="deal_id", how="left")
)

print("Filas finales:", len(df_final))
print(f"Con dni: {df_final['dni'].notna().sum() if 'dni' in df_final.columns else 0}")
print(f"Con deal_id: {df_final['deal_id'].notna().sum() if 'deal_id' in df_final.columns else 0}")
print(f"Con resolution: {df_final['resolution'].notna().sum() if 'resolution' in df_final.columns else 0}")

if "resolution" in df_final.columns:
    print("\nDistribucion de resolution:")
    print(df_final["resolution"].value_counts(dropna=False))

df_final.head()

Filas finales: 30233
Con dni: 28095
Con deal_id: 27444
Con resolution: 27435

Distribucion de resolution:
resolution
rejected       13975
approved        9118
preApproved     4342
NaN             2798
Name: count, dtype: int64


,date,lead_id,sales_channel,brand,model,color,price_net,totals,dni,deal_id,resolution
0,2026-05-16,6a089fc6c06ef2092619084e,dealer,TVS,Stryker 3V,Rojo,32990,4,OECI820607HYNLBS08,6a08a38dc23460deed4b756b,preApproved
1,2026-05-16,6a08c402decb411f4378bb24,dealer,Bajaj,Pulsar N 160,(none),37099,2,COGM070728MVZLRRA6,6a08c4bd5b60da0a86ffc755,approved
2,2026-05-18,6a08c402decb411f4378bb24,dealer,Bajaj,Pulsar N 160,(none),37099,2,COGM070728MVZLRRA6,6a08c4bd5b60da0a86ffc755,approved
3,2026-05-18,6a0b427a48fd1283afd97ea8,dealer,Loong,GR300,(none),55000,4,HERG911210HMCRYN04,6a0b4432f5e702191891eea2,approved
4,2026-05-18,6a0b4e3a86cb1ab4b5ccafb8,dealer,RYDER,RD-2.5 250,Rojo,41999,4,OECA620604MMCRRN05,6a0b66fb265b47bde9602a7f,approved


In [27]:
df_test = df_final[(df_final["resolution"] == "approved") & (df_final["date"].isin(["2026-05-19", "2026-05-18", "2026-05-19"])) & (df_final["sales_channel"] == "dealer")]

In [33]:
df_test

,date,lead_id,sales_channel,brand,model,color,price_net,totals,dni,deal_id,resolution
2,2026-05-18,6a08c402decb411f4378bb24,dealer,Bajaj,Pulsar N 160,(none),37099,2,COGM070728MVZLRRA6,6a08c4bd5b60da0a86ffc755,approved
3,2026-05-18,6a0b427a48fd1283afd97ea8,dealer,Loong,GR300,(none),55000,4,HERG911210HMCRYN04,6a0b4432f5e702191891eea2,approved
4,2026-05-18,6a0b4e3a86cb1ab4b5ccafb8,dealer,RYDER,RD-2.5 250,Rojo,41999,4,OECA620604MMCRRN05,6a0b66fb265b47bde9602a7f,approved
10,2026-05-18,6a0b7aaadb7e6cec60f4b084,dealer,TVS,Stryker 3V,Negro,32990,3,TEBM990307HYNLLR05,6a0b7c50961114d06ccb9703,approved
11,2026-05-18,6a0ba5977041087c6dbcc1d2,dealer,Bajaj,Pulsar N 250,(none),44099,3,COCJ060526HMNNRSA3,6a0ba6dcb85dabde89f42bc4,approved
...,...,...,...,...,...,...,...,...,...,...,...
10380,2026-05-19,6a0d11416361cb026107d3b9,dealer,Honda,XR 150L,(none),48590,1,RIRS751101MSLVVN00,6a0d12d5e761204256646079,approved
10387,2026-05-19,6a0d15216361cb026107d926,dealer,Omo Mobility,ATHENA X1,Azul,29999,1,OOVA650803HJCLZR05,6a0d15bbe76120425664672a,approved
10388,2026-05-19,6a0d15216361cb026107d926,dealer,TVS,TVS NTORQ 125 Race,Azul,14500,1,OOVA650803HJCLZR05,6a0d15bbe76120425664672a,approved
10389,2026-05-19,6a0d15fe831d9259141ba068,dealer,Vento,Storm 300 2.0,(none),39999,1,GARC071127HSPRMSA7,6a0d2195c0432a16b75529a2,approved


In [32]:
df_test[df_test["lead_id"] == ""]

,date,lead_id,sales_channel,brand,model,color,price_net,totals,dni,deal_id,resolution


In [28]:
df_test[df_test["deal_id"] == "6a06282e27193b335334d2b4"]

,date,lead_id,sales_channel,brand,model,color,price_net,totals,dni,deal_id,resolution


In [16]:
# out_path = f"cotizaciones_{fecha_corte.strftime('%Y%m%d')}.csv"
# df_final.to_csv(out_path, index=False)
# print(f"Guardado: {out_path}")